# DOOM on Google Colab

This project demonstrates how the classic DOOM engine can be executed in a cloud-based environment using a virtual display pipeline and browser-based streaming.

---

## Developer

**Zharif Athaya Andarfi**  
Full-Stack Developer & Computer Science Student  
GitHub: https://github.com/Zharif-110305

---

## Overview

This notebook utilizes a lightweight Linux graphical stack:

- Virtual framebuffer (Xvfb)
- VNC server (x11vnc)
- WebSocket bridge (noVNC / websockify)
- DOOM engine runtime (PrBoom+)

The objective is to provide a reproducible and interactive environment for running DOOM entirely within Google Colab.

---

## Acknowledgements & Credits

This project would not be possible without the contributions of the open-source community.  
Special thanks to the developers and maintainers of the following projects:

- PrBoom+ — a modern and enhanced DOOM engine source port
- Freedoom — a free and open-source IWAD replacement for DOOM
- Xvfb — X virtual framebuffer for headless display environments
- x11vnc — VNC server for real X displays
- noVNC — HTML5 VNC client for browser-based access
- websockify — WebSocket to TCP proxy

---

## Copyright & Licensing

All third-party software components used in this project are the property of their respective authors and are distributed under their own licenses.

- DOOM engine derivatives (e.g., PrBoom+) are based on id Software’s original DOOM source code, released under the GNU General Public License (GPL).
- Freedoom assets are distributed under free content licenses and are intended as a fully open replacement for proprietary DOOM data files.

This project serves as an educational and demonstrative implementation and does not claim ownership over any third-party software or assets.

---

## Disclaimer

This project is intended for educational and experimental purposes only.  
All components are used in accordance with their respective licenses.

---

Install Dependencies

In [ ]:
%%bash
set -e

apt-get update -qq
apt-get install -y -qq \
    xvfb \
    x11vnc \
    novnc \
    websockify \
    prboom-plus

echo "[INFO] Dependencies installed."

Config & Utilities

In [ ]:
import os
import socket
import time
import subprocess
import atexit

DISPLAY = ":99"
VNC_PORT = 5900
NOVNC_PORT = 6080
WAD_PATH = "/usr/share/games/doom/freedoom2.wad"
RESOLUTION = "800x600x16"
DOOM_WIDTH = "800"
DOOM_HEIGHT = "600"

processes = []

def register(p):
    processes.append(p)
    return p

def cleanup():
    for p in processes:
        if p.poll() is None:
            p.terminate()

atexit.register(cleanup)

def wait_for_port(port, timeout=10):
    start = time.time()
    while time.time() - start < timeout:
        with socket.socket() as s:
            if s.connect_ex(("localhost", port)) == 0:
                return True
        time.sleep(0.1)
    return False

print("[INFO] Environment ready.")

Validate WAD

In [ ]:
import os

if not os.path.exists(WAD_PATH):
    raise RuntimeError("WAD file not found. Ensure prboom-plus installed correctly.")

size_mb = os.path.getsize(WAD_PATH) / (1024 * 1024)

print("[INFO] WAD detected.")
print(f"[INFO] Path : {WAD_PATH}")
print(f"[INFO] Size : {size_mb:.2f} MB")

Start Xvfb

In [ ]:
os.environ["DISPLAY"] = DISPLAY

xvfb = register(subprocess.Popen(
    [
        "Xvfb", DISPLAY,
        "-screen", "0", RESOLUTION,
        "-ac",
        "-noreset"
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
))

time.sleep(1)

if xvfb.poll() is None:
    print("[INFO] Xvfb started.")
else:
    raise RuntimeError("Xvfb failed")

Start x11vnc

In [ ]:
vnc = register(subprocess.Popen(
    [
        "x11vnc",
        "-display", DISPLAY,
        "-nopw",
        "-listen", "localhost",
        "-forever",
        "-shared",
        "-ncache", "10",
        "-repeat"
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
))

if wait_for_port(VNC_PORT):
    print("[INFO] x11vnc started.")
else:
    raise RuntimeError("x11vnc failed")

Start noVNC

In [ ]:
novnc = register(subprocess.Popen(
    [
        "websockify",
        "--web=/usr/share/novnc/",
        str(NOVNC_PORT),
        f"localhost:{VNC_PORT}"
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
))

if wait_for_port(NOVNC_PORT):
    print("[INFO] noVNC started.")
else:
    raise RuntimeError("noVNC failed")

Run DOOM

In [ ]:
from google.colab.output import serve_kernel_port_as_iframe

doom = register(subprocess.Popen(
    [
        "/usr/games/prboom-plus",
        "-iwad", WAD_PATH,
        "-width", DOOM_WIDTH,
        "-height", DOOM_HEIGHT,
        "-vidmode", "sw",
        "-nosound"
        "-nomouse"
    ],
    env={**os.environ, "DISPLAY": DISPLAY},
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
))

time.sleep(2)

if doom.poll() is None:
    print("[INFO] DOOM started successfully.")
    print(f"[INFO] Access: http://localhost:{NOVNC_PORT}/vnc.html")

    serve_kernel_port_as_iframe(
        NOVNC_PORT,
        path="/vnc.html?autoconnect=true&resize=remote&scale=local&quality=4&compression=5",
        height=700
    )
else:
    raise RuntimeError("DOOM failed to start")